In [3]:

 
import pandas as pd
import numpy as np
import os

# Create output folder
os.makedirs("C:/Users/faizan/OneDrive/Desktop/Thiranex-Data-Science/data/processed", exist_ok=True)

# Load original datasets
customers = pd.read_csv(
    "C:/Users/faizan/OneDrive/Desktop/Thiranex-Data-Science/data/raw/olist_customers_dataset.csv",
    dtype={"customer_zip_code_prefix": str}
)
orders = pd.read_csv("C:/Users/faizan/OneDrive/Desktop/Thiranex-Data-Science/data/raw/olist_orders_dataset.csv")
order_items = pd.read_csv("C:/Users/faizan/OneDrive/Desktop/Thiranex-Data-Science/data/raw/olist_order_items_dataset.csv")
payments = pd.read_csv("C:/Users/faizan/OneDrive/Desktop/Thiranex-Data-Science/data/raw/olist_order_payments_dataset.csv")
reviews = pd.read_csv("C:/Users/faizan/OneDrive/Desktop/Thiranex-Data-Science/data/raw/olist_order_reviews_dataset.csv")

datasets = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews
}

# Remove exact duplicate rows
for name in datasets:
    datasets[name] = datasets[name].drop_duplicates()

# Clean customer text
customers = datasets["customers"]

for col in ["customer_city", "customer_state"]:
    customers[col] = customers[col].str.strip()

customers["customer_city"] = customers["customer_city"].str.lower()
customers["customer_state"] = customers["customer_state"].str.upper()

# Convert order date columns
orders = datasets["orders"]

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

# Clean payment data
payments = datasets["payments"]
payments["payment_type"] = payments["payment_type"].str.strip().str.lower()

# Inspect missing values after cleaning
print("MISSING VALUES AFTER CLEANING")
for name, df in datasets.items():
    print(f"\n{name}:")
    print(df.isnull().sum())

# Inspect payment value outliers using IQR
payments = datasets["payments"]
Q1 = payments["payment_value"].quantile(0.25)
Q3 = payments["payment_value"].quantile(0.75)
IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

outliers = payments[
    (payments["payment_value"] < lower_limit) |
    (payments["payment_value"] > upper_limit)
]

print("\nPayment value outliers:", len(outliers))

# Save cleaned datasets
for name, df in datasets.items():
    df.to_csv(f"C:/Users/faizan/OneDrive/Desktop/Thiranex-Data-Science/data/processed/{name}_cleaned.csv", index=False)

print("\nCLEANING COMPLETED")
print("Cleaned datasets saved in data/processed/")


MISSING VALUES AFTER CLEANING

customers:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

orders:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

order_items:
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

payments:
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

reviews:
review_id                      0
order_id                       0
review_score                   0
revi